# SAM 3D
In this demo, we will go over the fun features of SAM3d

## SAM 3D Objects
Create a 3D model from a single image using this demo:

[DEMO](https://aidemos.meta.com/segment-anything/editor/convert-image-to-3d)

### Gaussian splats
You can install a viewer inside your vs code extensions:

[Gaussian Viewer](https://marketplace.visualstudio.com/items?itemName=GaussianViewer.gaussian-viewer)

Below, we will read the raw data and play around by scaling the individual splats

In [5]:
# import the ply file
from plyfile import PlyData, PlyElement
import numpy as np

plyPath = r"C:\Users\jelle\Documents\DoctoraatLocal\generationtools\data\CompletionChairSplat.ply"

# read the raw ply data to extract the points
ply = PlyData.read(plyPath)
vertex = ply['vertex'].data

# Convert to a dict of numpy arrays
splat = {name: np.asarray(vertex[name]) for name in vertex.dtype.names}
print("Available fields:", splat.keys())

positions = np.stack([splat['x'], splat['y'], splat['z']], axis=1)
scales = np.stack([splat['scale_0'], splat['scale_1'], splat['scale_2']], axis=1)
colors = np.stack([splat['f_dc_0'],
                  splat['f_dc_1'],
                  splat['f_dc_2']], axis=1)



Available fields: dict_keys(['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3'])


### Visualise as a open3D pointcloud

In [6]:
import open3d as o3d
# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(positions)

if colors is not None:
    pcd.colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([pcd])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


### Scaling the splats

In [ ]:
import math
#-------- Parameters
startEndRange = np.array([0.35,0.65])  # Positions along the chosen axis normalised from 0 to 1
axis = 0  # Axis for the planes ('x', 'y', or 'z')
bbSize = 1 # the total size of the boundingbox (should be 1)
scaleFactor = 2 # Change this to other scaling values

#-------- Calculation
minVal = -bbSize/2.0 + startEndRange[0] * bbSize
maxVal = -bbSize/2.0 + startEndRange[1] * bbSize
distance = np.abs(maxVal-minVal)
newDist = distance * scaleFactor

newPositions = np.copy(positions)
newScales = np.copy(scales)
i = 0
for point in newPositions:
        # Move the plane to the specified axis
    if(point[axis] > minVal):
        # The point is further than the startplane
        if(point[axis] > maxVal):
            # The point is further than the end plane -> just move the max distance
            point[axis] += newDist - distance
        else:
            # The point is in between -> interpolate
            oldDist = point[axis]-minVal
            point[axis] = minVal + oldDist *scaleFactor 
            
            # scale local axes proportionally
            newScales[i] /= math.sqrt(math.sqrt(math.sqrt(scaleFactor)))
    i+=1

# Write back to the ply
splat['x'] = newPositions[:, 0]
splat['y'] = newPositions[:, 1]
splat['z'] = newPositions[:, 2]
splat['scale_0'] = newScales[:, 0]
splat['scale_1'] = newScales[:, 1]
splat['scale_2'] = newScales[:, 2]

# Reconstruct structured array
vertex_dtype = vertex.dtype
vertex_array = np.empty(len(splat['x']), dtype=vertex_dtype)

for name in vertex_dtype.names:
    vertex_array[name] = splat[name]

new_vertex = PlyElement.describe(vertex_array, 'vertex')

new_ply = PlyData([new_vertex], text=ply.text)
newPlyPath = plyPath[:-4] + "_axis_" + str(axis) + "_range_" + str(startEndRange[0]) + "-" + str(startEndRange[1]) + "_scale_" + str(scaleFactor) + ".ply"
new_ply.write(newPlyPath)